# SportsData.io NFL Stats Ingestion

Pull weekly NFL player statistics from **SportsData.io** API and land in bronze/silver Delta tables.

**Provider:** SportsData.io (https://sportsdata.io)  
**API Base:** `https://api.sportsdata.io/v3/nfl`  
**Free Tier:** 1,000 calls/month with real data  
**Authentication:** `Ocp-Apim-Subscription-Key` header

**Note:** This is different from API-Sports.io - requires separate subscription from SportsData.io

In [0]:
import requests
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# SportsData.io Configuration
API_KEY = "c78fb5a103d64f0b9e860ebc32c09401"  # From SportsDataIO dashboard
BASE_URL = "https://api.sportsdata.io/v3/nfl/stats"

# Configure week and season - Free trial may only include current season
WEEK = 18
SEASON = 2025  # Trying 2025 instead (free trial may not include historical data)

print(f"📅 Fetching stats for Week {WEEK}, Season {SEASON} from SportsData.io")
print(f"   Base URL: {BASE_URL}")
print(f"   Free tier: 1,000 calls/month")
print(f"   Note: Free trial may only include current/recent season data")

In [0]:
# Test SportsData.io authentication and endpoints
import requests
import json

print(f"Testing SportsData.io API access...\n")
print(f"API Key: {API_KEY[:8]}...{API_KEY[-4:]}\n")

# Test different authentication methods and endpoints
test_configs = [
    {
        "name": "v3/nfl/stats with Ocp-Apim-Subscription-Key",
        "url": f"https://api.sportsdata.io/v3/nfl/stats/json/PlayerGameStatsByWeek/{SEASON}/{WEEK}",
        "headers": {"Ocp-Apim-Subscription-Key": API_KEY}
    },
    {
        "name": "v3/nfl/scores with Ocp-Apim-Subscription-Key",
        "url": f"https://api.sportsdata.io/v3/nfl/scores/json/Scores/{SEASON}",
        "headers": {"Ocp-Apim-Subscription-Key": API_KEY}
    },
    {
        "name": "v3/nfl/stats with x-api-key",
        "url": f"https://api.sportsdata.io/v3/nfl/stats/json/PlayerGameStatsByWeek/{SEASON}/{WEEK}",
        "headers": {"x-api-key": API_KEY}
    },
    {
        "name": "URL parameter ?key=",
        "url": f"https://api.sportsdata.io/v3/nfl/stats/json/PlayerGameStatsByWeek/{SEASON}/{WEEK}?key={API_KEY}",
        "headers": {}
    },
]

print("Testing endpoints:\n")
working_config = None

for config in test_configs:
    try:
        response = requests.get(config["url"], headers=config["headers"], timeout=10)
        status = response.status_code
        
        print(f"  {config['name']}: {status}")
        
        if status == 200:
            print(f"    ✓ SUCCESS!")
            working_config = config
            break
        elif status == 401:
            print(f"    ✗ Unauthorized")
        elif status == 403:
            print(f"    ✗ Forbidden - endpoint not included in subscription")
        elif status == 404:
            print(f"    ? Not found - no data for this week/season")
        else:
            print(f"    ? Status {status}")
            
        # Show error details for first failure
        if status != 200 and config == test_configs[0]:
            print(f"    Response: {response.text[:200]}")
            
    except Exception as e:
        print(f"  {config['name']}: ERROR - {str(e)[:80]}")
    print()

print("="*60)

if working_config:
    print(f"\n✓ Found working configuration: {working_config['name']}\n")
    
    # Fetch the actual data
    response = requests.get(working_config["url"], headers=working_config["headers"], timeout=30)
    stats_data = response.json()
    
    print(f"✓ Fetched {len(stats_data)} player records\n")
    
    if stats_data:
        print("Sample player record:")
        print(json.dumps(stats_data[0], indent=2)[:1500])
        print("\n...\n")
    
    # Convert to DataFrame
    rows = []
    for player_stat in stats_data:
        player_id = str(player_stat.get('PlayerID', ''))
        fantasy_points = float(player_stat.get('FantasyPointsPPR', 0) or 0)
        
        rows.append(
            Row(
                player_id=player_id,
                week=WEEK,
                season=SEASON,
                fantasy_points=fantasy_points,
                stats=json.dumps(player_stat),
                source='sportsdata'
            )
        )
    
    stats_df = spark.createDataFrame(rows)
    print(f"✓ Created DataFrame with {stats_df.count()} player records\n")
    display(stats_df.limit(20))
    
else:
    print("\n⚠️  No working configuration found\n")
    print("Possible issues:")
    print("  1. Free trial needs manual activation - check your email")
    print("  2. Need to enable NFL Stats feed in dashboard")
    print("  3. API key needs 15-30 minutes to activate")
    print("  4. May need to use SportsDataIO GRid platform instead")
    print("\nNext steps:")
    print("  1. Check email for activation link")
    print("  2. Visit https://sportsdata.io/members/subscriptions")
    print("  3. Verify 'NFL Stats' feed is enabled")
    print("  4. Wait 15-30 mins and try again")

In [0]:
# Write to bronze table using MERGE
if 'stats_df' in locals():
    bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp())
    
    # Cast week/season to int to match table schema
    bronze_df = bronze_df \
        .withColumn("week", F.col("week").cast('int')) \
        .withColumn("season", F.col("season").cast('int'))
    
    # Create temp view for merge
    bronze_df.createOrReplaceTempView("sportsdata_bronze_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.bronze_weekly_stats AS target
      USING sportsdata_bronze_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
    """)
    
    print(f"✓ Merged {bronze_df.count()} records from SportsData.io into bronze_weekly_stats")
else:
    print("⚠️  No data to write - please run Cell 3 first")

In [0]:
# Transform for silver
if 'bronze_df' in locals():
    silver_df = (
        bronze_df
        .select(
            F.col("player_id").cast("string"),
            F.col("week").cast("int"),
            F.col("season").cast("int"),
            F.col("fantasy_points").cast("double"),
            F.col("stats").cast("string"),
            F.col("ingested_at"),
        )
        .dropDuplicates(["player_id", "week", "season"])
    )
    
    # Create temp view for merge
    silver_df.createOrReplaceTempView("sportsdata_silver_updates")
    
    # Perform MERGE operation
    spark.sql("""
      MERGE INTO main.fantasai.silver_weekly_stats AS target
      USING sportsdata_silver_updates AS source
      ON target.player_id = source.player_id 
        AND target.week = source.week 
        AND target.season = source.season
      WHEN MATCHED THEN
        UPDATE SET
          target.fantasy_points = source.fantasy_points,
          target.stats = source.stats,
          target.ingested_at = source.ingested_at
      WHEN NOT MATCHED THEN
        INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
        VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
    """)
    
    print(f"✓ Merged {silver_df.count()} records from SportsData.io into silver_weekly_stats")
else:
    print("⚠️  No data to write - please run Cell 4 first")